<a href="https://colab.research.google.com/github/TAlkam/NRD_2017/blob/main/NRD_2017_for_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pyreadstat xgboost shap scikit-learn statsmodels


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose your .dta (or .zip)


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_stata(list(uploaded.keys())[0])

dx_cols = sorted([c for c in df.columns if c.startswith("i10_dx")])

def any_icd_prefix_fast(df, cols, prefixes):
    prefixes = tuple(p.upper() for p in prefixes)
    hit = np.zeros(len(df), dtype=bool)
    for c in cols:
        s = df[c].astype("string").str.upper()
        hit |= s.str.startswith(prefixes, na=False).to_numpy()
    return pd.Series(hit, index=df.index)

df["adrd_any"] = any_icd_prefix_fast(df, dx_cols, ["G30","F01","F02","F03","G31"])
df["ad_only"]  = any_icd_prefix_fast(df, dx_cols, ["G30"])
print("ADRD prevalence:", df["adrd_any"].mean())

In [ ]:
# Outcomes
df["died"] = pd.to_numeric(df.get("died", 0), errors="coerce").fillna(0).astype(int)
df["los"]  = pd.to_numeric(df.get("los", np.nan), errors="coerce")

# Prolonged LOS definition (choose one)
q75 = df.loc[df["adrd_any"] & df["los"].notna(), "los"].quantile(0.75)
df["plos"] = ((df["los"] >= q75) & df["los"].notna()).astype(int)

print("PLOS cutoff (75th pct) =", q75)


In [ ]:
flag_map = {
    "sepsis": ["A40","A41"],
    "pneumonia": ["J12","J13","J14","J15","J16","J17","J18"],
    "uti": ["N39"],
    "aki": ["N17"],
    "chf": ["I50"],
    "copd": ["J44"],
    "ckd": ["N18"],
    "stroke": ["I60","I61","I62","I63"],
    "delirium": ["F05"]
}
for name, pref in flag_map.items():
    df[name] = any_icd_prefix_fast(df, dx_cols, pref)

# Choose predictors
feat_candidates = [
    "age","female","pay1","zipinc_qrtl","pl_nchs","aweekend","elective","hcup_ed",
    "i10_ndx","i10_npr",  # burden proxies
    "sepsis","pneumonia","uti","aki","chf","copd","ckd","stroke","delirium",
    "ad_only"
]
feat_cols = [c for c in feat_candidates if c in df.columns]

# Cohort: ADRD admissions
dat = df[df["adrd_any"]].copy()
print("ADRD N:", len(dat), "Deaths:", int(dat["died"].sum()), "PLOS events:", int(dat["plos"].sum()))


In [ ]:
# ============================================================
# Figure 2 (publishable): Model A vs Model B ROC (5-fold CV)
# Panel A: Mean ROC curves
# Panel B: Inset with mean AUROC ± SD
# ============================================================

!pip -q install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# --------------------------
# 0) CONFIG
# --------------------------
DF_NAME = "dat"          # your dataframe variable name
TARGET = "died"        # outcome column (0/1)
RANDOM_STATE = 42
N_SPLITS = 5

# Post-admission proxies
LOS_COL  = "los"
PROC_COL = "i10_npr"
CHG_COL  = "totchg"    # or "totchg_10k" if you created that

# Candidate predictors (edit to match your notebook; non-existing columns are auto-dropped)
base_features = [
    "age", "female",
    "sepsis", "aki", "stroke", "uti", "chf", "ckd", "pneumonia", "delirium", "copd",
    "i10_ndx",                 # number of diagnoses
    "aweekend", "elective",
    "pay1", "zipinc_qrtl", "pl_nchs", "hcup_ed",
]

# --------------------------
# 1) LOAD DATA
# --------------------------
d = globals().get(DF_NAME, None)
if d is None:
    raise ValueError(f"DataFrame `{DF_NAME}` not found. Set DF_NAME to your dataframe variable name.")

dd = d.dropna(subset=[TARGET]).copy()
y = dd[TARGET].astype(int).values

# Keep only existing columns
base_features = [c for c in base_features if c in dd.columns]

# Model A: admission-only (exclude post-admission proxies)
features_A = [c for c in base_features if c not in [LOS_COL, PROC_COL, CHG_COL]]

# Model B: full inpatient course (include proxies if present)
features_B = list(dict.fromkeys(base_features + [LOS_COL, PROC_COL, CHG_COL]))
features_B = [c for c in features_B if c in dd.columns]

print("Model A features:", features_A)
print("Model B features:", features_B)

# --------------------------
# 2) PREPROCESSOR + MODEL
# --------------------------
def make_preprocessor(df, features):
    X = df[features]
    cat_cols = [c for c in features if (X[c].dtype == "object") or str(X[c].dtype).startswith("category")]
    num_cols = [c for c in features if c not in cat_cols]

    numeric = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric, num_cols),
            ("cat", categorical, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )
    return pre

def make_model():
    return XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

# --------------------------
# 3) MEAN ROC ACROSS FOLDS
# --------------------------
def cv_mean_roc(df, y, features, label):
    Xraw = df[features].copy()
    pre = make_preprocessor(df, features)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    aucs = []
    fpr_grid = np.linspace(0, 1, 401)
    tprs_interp = []

    for fold, (tr, te) in enumerate(skf.split(Xraw, y), 1):
        Xtr, Xte = Xraw.iloc[tr], Xraw.iloc[te]
        ytr, yte = y[tr], y[te]

        Xtr_t = pre.fit_transform(Xtr)
        Xte_t = pre.transform(Xte)

        clf = make_model()
        clf.fit(Xtr_t, ytr)

        p = clf.predict_proba(Xte_t)[:, 1]
        auc = roc_auc_score(yte, p)
        aucs.append(auc)

        fpr, tpr, _ = roc_curve(yte, p)
        # interpolate tpr over the common fpr grid
        tpr_i = np.interp(fpr_grid, fpr, tpr)
        tpr_i[0] = 0.0
        tprs_interp.append(tpr_i)

        print(f"{label} Fold {fold}: AUROC={auc:.3f}")

    tprs_interp = np.vstack(tprs_interp)
    mean_tpr = tprs_interp.mean(axis=0)
    mean_tpr[-1] = 1.0
    sd_tpr = tprs_interp.std(axis=0, ddof=1)

    return {
        "label": label,
        "fpr_grid": fpr_grid,
        "mean_tpr": mean_tpr,
        "sd_tpr": sd_tpr,
        "aucs": aucs,
        "auc_mean": float(np.mean(aucs)),
        "auc_sd": float(np.std(aucs, ddof=1)),
    }

resA = cv_mean_roc(dd, y, features_A, "Model A (Admission-only)")
resB = cv_mean_roc(dd, y, features_B, "Model B (Inpatient-course)")

print("\nSummary:")
print(f"Model A AUROC: {resA['auc_mean']:.3f} ± {resA['auc_sd']:.3f}")
print(f"Model B AUROC: {resB['auc_mean']:.3f} ± {resB['auc_sd']:.3f}")

# --------------------------
# 4) PLOT FIGURE 2 (PUBLISHABLE)
# --------------------------
plt.figure(figsize=(7.2, 6.2))

# Diagonal reference
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

# Mean ROC curves
plt.plot(resA["fpr_grid"], resA["mean_tpr"], linewidth=2, label=f"{resA['label']}")
plt.plot(resB["fpr_grid"], resB["mean_tpr"], linewidth=2, label=f"{resB['label']}")

# Optional: show variability bands (can remove if you want a cleaner figure)
plt.fill_between(resA["fpr_grid"],
                 np.clip(resA["mean_tpr"] - resA["sd_tpr"], 0, 1),
                 np.clip(resA["mean_tpr"] + resA["sd_tpr"], 0, 1),
                 alpha=0.15)
plt.fill_between(resB["fpr_grid"],
                 np.clip(resB["mean_tpr"] - resB["sd_tpr"], 0, 1),
                 np.clip(resB["mean_tpr"] + resB["sd_tpr"], 0, 1),
                 alpha=0.15)

plt.xlabel("1 − Specificity (False Positive Rate)")
plt.ylabel("Sensitivity (True Positive Rate)")
plt.title("Figure 2. Model A vs Model B: Discrimination for In-hospital Mortality")

plt.legend(loc="lower right", frameon=True)

# Inset-style text box (Panel B info)
txt = (
    "5-fold cross-validation\n"
    f"Model A AUROC = {resA['auc_mean']:.3f} ± {resA['auc_sd']:.3f}\n"
    f"Model B AUROC = {resB['auc_mean']:.3f} ± {resB['auc_sd']:.3f}"
)
plt.gca().text(
    0.05, 0.20, txt,
    transform=plt.gca().transAxes,
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9)
)

plt.tight_layout()

# Save high-res
out_png = "Figure2_ModelA_vs_ModelB_ROC.png"
plt.savefig(out_png, dpi=600, bbox_inches="tight")
plt.show()

print(f"Saved: {out_png}")

In [ ]:
# ============================================================
# Model A vs Model B + SHAP (5-fold CV) — Colab-ready (PUBLISHABLE LABELS)
# - Model A: admission-only (excludes LOS, procedures, charges)
# - Model B: full inpatient course (includes LOS, I10_NPR, TOTCHG)
# Outputs:
#   * Fold AUROC + mean/SD
#   * SHAP summary (beeswarm) + mean(|SHAP|) bar per model (with publishable labels)
#   * SHAP stability table (with publishable labels)
# Notes:
#   - Assumes your dataframe is named d (or set DF_NAME below).
#   - Handles one-hot expansion: e.g., "pay1_2" -> "Primary payer: Medicare"
# ============================================================

!pip -q install xgboost shap

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

import shap
import matplotlib.pyplot as plt

# --------------------------
# 0) CONFIG
# --------------------------
DF_NAME = "d"          # your dataframe variable name
TARGET = "died"        # outcome column (0/1)
RANDOM_STATE = 42
N_SPLITS = 5

# Define columns that are post-admission proxies
LOS_COL = "los"        # length of stay column name
PROC_COL = "i10_npr"   # number of procedures column name
CHG_COL = "totchg"     # total charges column name
# If you modeled total charges per 10k, set CHG_COL="totchg_10k"

# --------------------------
# 0b) PUBLISHABLE LABELS (edit as desired)
# --------------------------
# Base (non one-hot) variable labels
BASE_LABELS = {
    "age": "Age (years)",
    "female": "Female sex",
    "sepsis": "Sepsis",
    "aki": "Acute kidney injury",
    "stroke": "Stroke",
    "uti": "Urinary tract infection",
    "chf": "Congestive heart failure",
    "ckd": "Chronic kidney disease",
    "pneumonia": "Pneumonia",
    "delirium": "Delirium",
    "copd": "Chronic obstructive pulmonary disease",
    "i10_ndx": "Number of diagnoses (ICD-10-CM)",
    "aweekend": "Weekend admission",
    "elective": "Elective admission",
    "pay1": "Primary payer",
    "zipinc_qrtl": "ZIP income quartile",
    "pl_nchs": "Urban–rural residence (NCHS category)",
    "hcup_ed": "Emergency department involvement",
    "ad_only": "Alzheimer’s disease only (indicator)",
    "los": "Length of stay (days)",
    "i10_npr": "Number of procedures (ICD-10-PCS)",
    "totchg": "Total hospital charges (USD)",
    "totchg_10k": "Total hospital charges (per $10,000)",
}

# For one-hot expanded categories: map raw codes -> readable category names
# (If your coding differs, edit these dictionaries.)
PAY1_CATS = {
    "1": "Medicare",
    "2": "Medicaid",
    "3": "Private insurance",
    "4": "Self-pay",
    "5": "No charge",
    "6": "Other",
    "nan": "Missing/unknown"
}
ZIPINC_CATS = {
    "1": "1st quartile (lowest)",
    "2": "2nd quartile",
    "3": "3rd quartile",
    "4": "4th quartile (highest)",
    "nan": "Missing/unknown"
}
PL_NCHS_CATS = {
    "1": "Large central metro",
    "2": "Large fringe metro",
    "3": "Medium metro",
    "4": "Small metro",
    "5": "Micropolitan",
    "6": "Noncore/rural",
    "nan": "Missing/unknown"
}
HCUP_ED_CATS = {
    # adjust if yours differs
    "0": "No ED involvement",
    "1": "ED involvement",
    "2": "ED involvement (category 2)",
    "3": "ED involvement (category 3)",
    "4": "ED involvement (category 4)",
    "nan": "Missing/unknown"
}

def _nice_fallback(name: str) -> str:
    s = str(name).replace("_", " ").strip()
    return s[:1].upper() + s[1:] if s else s

def make_publishable_feature_names(raw_feature_names):
    """
    raw_feature_names come from ColumnTransformer.get_feature_names_out().
    For one-hot cols, we expect names like:
      pay1_2, zipinc_qrtl_4, pl_nchs_2, hcup_ed_1, elective_1.0, aweekend_1.0, female_1.0, ad_only_1.0, etc.
    For numeric cols, names like: age, i10_ndx, los, i10_npr, totchg
    """
    out = []
    for f in raw_feature_names:
        f = str(f)

        # Handle common one-hot patterns: var_level
        if "_" in f:
            var, lvl = f.split("_", 1)

            # payer
            if var == "pay1":
                label = f"{BASE_LABELS.get('pay1','Primary payer')}: {PAY1_CATS.get(lvl, lvl)}"
                out.append(label); continue

            # ZIP income quartile
            if var == "zipinc" and lvl.startswith("qrtl"):
                # Sometimes OneHotEncoder might output "zipinc_qrtl_2" already as var="zipinc" lvl="qrtl_2"
                # Try to parse last token as the level
                parts = f.split("_")
                lvl2 = parts[-1]
                label = f"{BASE_LABELS.get('zipinc_qrtl','ZIP income quartile')}: {ZIPINC_CATS.get(lvl2, lvl2)}"
                out.append(label); continue

            if var == "zipinc_qrtl":
                label = f"{BASE_LABELS.get('zipinc_qrtl','ZIP income quartile')}: {ZIPINC_CATS.get(lvl, lvl)}"
                out.append(label); continue

            # NCHS
            if var == "pl" and lvl.startswith("nchs"):
                parts = f.split("_")
                lvl2 = parts[-1]
                label = f"{BASE_LABELS.get('pl_nchs','Urban–rural residence (NCHS category)')}: {PL_NCHS_CATS.get(lvl2, lvl2)}"
                out.append(label); continue

            if var == "pl_nchs":
                label = f"{BASE_LABELS.get('pl_nchs','Urban–rural residence (NCHS category)')}: {PL_NCHS_CATS.get(lvl, lvl)}"
                out.append(label); continue

            # HCUP ED
            if var == "hcup" and lvl.startswith("ed"):
                parts = f.split("_")
                lvl2 = parts[-1]
                label = f"{BASE_LABELS.get('hcup_ed','Emergency department involvement')}: {HCUP_ED_CATS.get(lvl2, lvl2)}"
                out.append(label); continue

            if var == "hcup_ed":
                label = f"{BASE_LABELS.get('hcup_ed','Emergency department involvement')}: {HCUP_ED_CATS.get(lvl, lvl)}"
                out.append(label); continue

            # binary indicators (0/1) one-hot expanded
            if var in ["female", "aweekend", "elective", "ad_only"]:
                base = BASE_LABELS.get(var, _nice_fallback(var))
                if lvl in ["1", "1.0", "True", "true"]:
                    out.append(base)  # show the "Yes" indicator as the feature
                elif lvl in ["0", "0.0", "False", "false"]:
                    # Often redundant; but keep for completeness if it appears
                    out.append(f"{base} (ref: no)")
                else:
                    out.append(f"{base}: {lvl}")
                continue

            # Otherwise fall back to base label if exists
            if f in BASE_LABELS:
                out.append(BASE_LABELS[f]); continue
            if var in BASE_LABELS:
                out.append(f"{BASE_LABELS[var]}: {lvl}"); continue

        # Non one-hot: numeric or already clean
        out.append(BASE_LABELS.get(f, _nice_fallback(f)))

    return np.array(out, dtype=object)

# --------------------------
# 1) LOAD DF
# --------------------------
d = globals().get(DF_NAME, None)
if d is None:
    raise ValueError(f"DataFrame `{DF_NAME}` not found. Set DF_NAME to your dataframe variable name.")

dd = d.dropna(subset=[TARGET]).copy()
y = dd[TARGET].astype(int).values

# --------------------------
# 2) FEATURE SETS
# --------------------------
base_features = [
    "age", "female",
    "sepsis", "aki", "stroke", "uti", "chf", "ckd", "pneumonia", "delirium", "copd",
    "i10_ndx",
    "aweekend", "elective",
    "pay1", "zipinc_qrtl", "pl_nchs", "hcup_ed",
    "ad_only"
]
base_features = [c for c in base_features if c in dd.columns]

features_A = [c for c in base_features if c not in [LOS_COL, PROC_COL, CHG_COL]]
features_B = list(dict.fromkeys(base_features + [LOS_COL, PROC_COL, CHG_COL]))
features_B = [c for c in features_B if c in dd.columns]

print("Model A features:", features_A)
print("Model B features:", features_B)

# --------------------------
# 3) PREPROCESSING PIPELINE
# --------------------------
def make_preprocessor(df, features):
    X = df[features]
    cat_cols = [c for c in features if X[c].dtype == "object" or str(X[c].dtype).startswith("category")]
    num_cols = [c for c in features if c not in cat_cols]

    numeric = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric, num_cols),
            ("cat", categorical, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )
    return pre, num_cols, cat_cols

def get_feature_names(preprocessor):
    return preprocessor.get_feature_names_out()

# --------------------------
# 4) MODEL
# --------------------------
def make_model():
    return XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

# --------------------------
# 5) CV + SHAP + STABILITY (WITH PUBLISHABLE LABELS)
# --------------------------
def run_cv_shap(df, y, features, model_name="Model"):
    Xraw = df[features].copy()
    pre, num_cols, cat_cols = make_preprocessor(df, features)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    aucs = []
    top10_per_fold = []
    shap_means_per_fold = []

    pooled_shap = []
    pooled_X = []

    for fold, (tr, te) in enumerate(skf.split(Xraw, y), 1):
        Xtr, Xte = Xraw.iloc[tr], Xraw.iloc[te]
        ytr, yte = y[tr], y[te]

        Xtr_t = pre.fit_transform(Xtr)
        Xte_t = pre.transform(Xte)

        feat_names_raw = get_feature_names(pre)
        feat_names_pub = make_publishable_feature_names(feat_names_raw)

        clf = make_model()
        clf.fit(Xtr_t, ytr)

        p = clf.predict_proba(Xte_t)[:, 1]
        auc = roc_auc_score(yte, p)
        aucs.append(auc)

        explainer = shap.TreeExplainer(clf)
        shap_vals = explainer.shap_values(Xte_t)  # (n, p)

        mean_abs = np.mean(np.abs(shap_vals), axis=0)
        order = np.argsort(mean_abs)[::-1]
        top10_idx = order[:10]
        top10_feats = [feat_names_pub[i] for i in top10_idx]

        top10_per_fold.append(top10_feats)
        shap_means_per_fold.append(pd.Series(mean_abs, index=feat_names_pub))

        if Xte_t.shape[0] > 2000:
            keep = np.random.RandomState(RANDOM_STATE + fold).choice(Xte_t.shape[0], 2000, replace=False)
            pooled_shap.append(shap_vals[keep])
            pooled_X.append(Xte_t[keep])
        else:
            pooled_shap.append(shap_vals)
            pooled_X.append(Xte_t)

        print(f"{model_name} Fold {fold}: AUROC={auc:.3f} | Top features: {', '.join(top10_feats[:5])} ...")

    auc_mean = float(np.mean(aucs))
    auc_sd = float(np.std(aucs, ddof=1))

    # Stability table
    all_feats = sorted(set([f for fold_feats in top10_per_fold for f in fold_feats]))
    appear_counts = {f: 0 for f in all_feats}
    rank_sums = {f: 0.0 for f in all_feats}

    for fold_feats in top10_per_fold:
        for r, f in enumerate(fold_feats, start=1):
            appear_counts[f] += 1
            rank_sums[f] += r

    stab = pd.DataFrame({
        "appear_top10_folds": pd.Series(appear_counts),
        "avg_rank_when_in_top10": pd.Series({f: (rank_sums[f] / appear_counts[f]) for f in all_feats})
    }).sort_values(["appear_top10_folds", "avg_rank_when_in_top10"], ascending=[False, True])

    mean_abs_all = pd.concat(shap_means_per_fold, axis=1).mean(axis=1).sort_values(ascending=False)

    pooled_shap_arr = np.vstack(pooled_shap)
    pooled_X_arr = np.vstack(pooled_X)

    # ---- Plots (publishable labels) ----
    shap.summary_plot(
        pooled_shap_arr,
        features=pooled_X_arr,
        feature_names=mean_abs_all.index.tolist(),  # publishable labels
        show=False,
        max_display=20
    )
    plt.xlabel("SHAP value (impact on model output)")
    plt.title(f"{model_name}: SHAP summary (beeswarm)")
    plt.tight_layout()
    plt.show()

    top20 = mean_abs_all.head(20)
    plt.figure(figsize=(8, 6))
    plt.barh(top20.index[::-1], top20.values[::-1])
    plt.xlabel("mean(|SHAP value|)")
    plt.title(f"{model_name}: Mean absolute SHAP importance (top 20)")
    plt.tight_layout()
    plt.show()

    out = {
        "aucs": aucs,
        "auc_mean": auc_mean,
        "auc_sd": auc_sd,
        "stability": stab,
        "mean_abs_shap": mean_abs_all,
        "top10_per_fold": top10_per_fold
    }
    return out

# --------------------------
# 6) RUN BOTH MODELS
# --------------------------
resA = run_cv_shap(dd, y, features_A, model_name="Model A (admission-only)")
print("\nAUROC Model A:", {"mean": resA["auc_mean"], "sd": resA["auc_sd"], "folds": resA["aucs"]})
display(resA["stability"].head(15))

resB = run_cv_shap(dd, y, features_B, model_name="Model B (full inpatient course)")
print("\nAUROC Model B:", {"mean": resB["auc_mean"], "sd": resB["auc_sd"], "folds": resB["aucs"]})
display(resB["stability"].head(15))

# --------------------------
# 7) OPTIONAL: EXPORT STABILITY TABLES
# --------------------------
resA["stability"].to_csv("shap_stability_modelA_publishable.csv")
resB["stability"].to_csv("shap_stability_modelB_publishable.csv")
print("\nSaved: shap_stability_modelA_publishable.csv, shap_stability_modelB_publishable.csv")

In [ ]:
import pandas as pd
[c for c,v in globals().items() if isinstance(v, pd.DataFrame)][:30]

In [ ]:
DF_NAME = "df"

In [ ]:
# =========================
# Publishable SHAP plots (Model A + Model B) — no abbreviations
# Copy–paste this cell and run AFTER your SHAP computation cell.
# =========================

import matplotlib.pyplot as plt
import pandas as pd
import shap

# --------- 1) AUTO-FIND your variables (edit here ONLY if your names differ) ---------
def _first_existing(*names):
    g = globals()
    for n in names:
        if n in g and g[n] is not None:
            return g[n], n
    return None, None

X_A, X_A_name = _first_existing("X_A", "XA", "X_adm", "X_admission", "X_modelA")
X_B, X_B_name = _first_existing("X_B", "XB", "X_full", "X_inpatient", "X_modelB")

shap_values_A, shapA_name = _first_existing("shap_values_A", "shapA", "sv_A", "SV_A", "shap_vals_A")
shap_values_B, shapB_name = _first_existing("shap_values_B", "shapB", "sv_B", "SV_B", "shap_vals_B")

missing = []
if X_A is None: missing.append("X_A (Model A feature matrix)")
if X_B is None: missing.append("X_B (Model B feature matrix)")
if shap_values_A is None: missing.append("shap_values_A (Model A SHAP values)")
if shap_values_B is None: missing.append("shap_values_B (Model B SHAP values)")

if missing:
    raise ValueError(
        "I couldn't find these objects in memory:\n"
        + "\n".join([f" - {m}" for m in missing]) +
        "\n\nFix: rename your variables to X_A, X_B, shap_values_A, shap_values_B "
        "OR edit the _first_existing(...) name lists at the top of this cell."
    )

print(f"Using: {X_A_name=}, {shapA_name=}, {X_B_name=}, {shapB_name=}")

# Ensure pandas DataFrames
if not isinstance(X_A, pd.DataFrame): X_A = pd.DataFrame(X_A)
if not isinstance(X_B, pd.DataFrame): X_B = pd.DataFrame(X_B)

# --------- 2) FEATURE LABELS (abbreviation -> journal-friendly label) ---------
FEATURE_LABELS = {
    "sepsis": "Sepsis",
    "aki": "Acute kidney injury",
    "stroke": "Stroke",
    "chf": "Congestive heart failure",
    "uti": "Urinary tract infection",
    "ckd": "Chronic kidney disease",
    "pneumonia": "Pneumonia",
    "copd": "Chronic obstructive pulmonary disease",
    "delirium": "Delirium",
    "female": "Female sex",
    "age": "Age (years)",
    "aweekend": "Weekend admission",
    "elective": "Elective admission",
    "hcup_ed": "Emergency department involvement",
    "pl_nchs": "Urban–rural residence (NCHS category)",
    "zipinc_qrtl": "ZIP income quartile",
    "pay1": "Primary payer",
    "ad_only": "Alzheimer’s disease only (indicator)",

    # inpatient-course proxies (Model B)
    "los": "Length of stay (days)",
    "totchg": "Total hospital charges",
    "totchg_10k": "Total charges (per $10,000)",
    "i10_npr": "Number of procedures (ICD-10-PCS)",
    "i10_ndx": "Number of diagnoses (ICD-10-CM)",
}

def _nice_fallback(name: str) -> str:
    """Fallback label if a column isn't in FEATURE_LABELS."""
    s = str(name).replace("_", " ").strip()
    return s[:1].upper() + s[1:]

def relabel_X(X: pd.DataFrame) -> pd.DataFrame:
    Xp = X.copy()
    Xp.columns = [FEATURE_LABELS.get(c, _nice_fallback(c)) for c in Xp.columns]
    return Xp

# --------- 3) PLOTTERS (beeswarm + bar) ---------
def shap_beeswarm(shap_values, X: pd.DataFrame, title: str,
                  max_display: int = 20, figsize=(7.5, 9.0), savepath: str | None = None):
    Xp = relabel_X(X)
    plt.figure(figsize=figsize)
    shap.summary_plot(shap_values, Xp, plot_type="dot", max_display=max_display, show=False)
    plt.xlabel("SHAP value (impact on model output)", fontsize=11)
    plt.title(title, fontsize=13)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=600, bbox_inches="tight")
    plt.show()

def shap_bar(shap_values, X: pd.DataFrame, title: str,
             max_display: int = 20, figsize=(7.5, 9.0), savepath: str | None = None):
    Xp = relabel_X(X)
    plt.figure(figsize=figsize)
    shap.summary_plot(shap_values, Xp, plot_type="bar", max_display=max_display, show=False)
    plt.xlabel("mean(|SHAP value|) (average impact magnitude)", fontsize=11)
    plt.title(title, fontsize=13)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=600, bbox_inches="tight")
    plt.show()

# --------- 4) MAKE PUBLISHABLE FIGURES ---------
# Beeswarm
shap_beeswarm(
    shap_values_A, X_A,
    title="Model A (admission-only): SHAP summary (beeswarm)",
    max_display=20,
    savepath="Figure_SHAP_ModelA_beeswarm.png"  # set None if you don't want saving
)

shap_beeswarm(
    shap_values_B, X_B,
    title="Model B (full inpatient course): SHAP summary (beeswarm)",
    max_display=20,
    savepath="Figure_SHAP_ModelB_beeswarm.png"
)

# Optional: Bar plots (often nicer for print)
shap_bar(
    shap_values_A, X_A,
    title="Model A (admission-only): SHAP feature importance",
    max_display=20,
    savepath="Figure_SHAP_ModelA_bar.png"
)

shap_bar(
    shap_values_B, X_B,
    title="Model B (full inpatient course): SHAP feature importance",
    max_display=20,
    savepath="Figure_SHAP_ModelB_bar.png"
)

print("Done. Saved PNGs in the current working directory (if savepath was not None).")

In [ ]:
!pip -q install xgboost shap scikit-learn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import shap

X = dat[feat_cols].copy()
y = dat["died"].astype(int).copy()

cat_cols = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith("string")]
num_cols = [c for c in X.columns if c not in cat_cols]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

clf = XGBClassifier(
    n_estimators=600, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, min_child_weight=1,
    objective="binary:logistic", eval_metric="logloss",
    tree_method="hist", random_state=42,
    scale_pos_weight=scale_pos_weight
)

pipe = Pipeline([("pre", pre), ("model", clf)])
pipe.fit(X_train, y_train)

p = pipe.predict_proba(X_test)[:,1]
print("Death AUROC:", roc_auc_score(y_test, p))
print("Death AUPRC:", average_precision_score(y_test, p), "Test positives:", int(y_test.sum()))

# SHAP
X_test_mat = pipe.named_steps["pre"].transform(X_test)
explainer = shap.TreeExplainer(pipe.named_steps["model"])
shap_vals = explainer.shap_values(X_test_mat)

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
cat_names = ohe.get_feature_names_out(cat_cols) if len(cat_cols) else np.array([])
feature_names = np.concatenate([np.array(num_cols), cat_names])

shap.summary_plot(shap_vals, X_test_mat, feature_names=feature_names, show=True)
shap.summary_plot(shap_vals, X_test_mat, feature_names=feature_names, plot_type="bar", show=True)


In [ ]:
import numpy as np
import textwrap

# 1) Map raw variable names -> publishable labels
pretty_map = {
    "age": "Age (years)",
    "female": "Female sex",
    "sepsis": "Sepsis",
    "aki": "Acute kidney injury",
    "chf": "Congestive heart failure",
    "ckd": "Chronic kidney disease",
    "uti": "Urinary tract infection",
    "stroke": "Stroke",
    "pneumonia": "Pneumonia",
    "aweekend": "Weekend admission",
    "i10_ndx": "Number of diagnoses (ICD-10-CM)",
    "i10_npr": "Number of procedures (ICD-10-PCS)",
    "zipinc_qrtl_1": "ZIP income quartile: 1 (lowest)",
    "zipinc_qrtl_3": "ZIP income quartile: 3",
    "pl_nchs_1": "Patient location (NCHS): Category 1",
    "pl_nchs_2": "Patient location (NCHS): Category 2",
    "pl_nchs_3": "Patient location (NCHS): Category 3",
    "pay1_1": "Primary payer: Category 1",
    "ad_only": "Alzheimer’s disease only (indicator)",
    "hcup_ed": "ED-related indicator",
}

# 2) If you have one-hot names like "PAY1_Medicare", convert to "Primary payer: Medicare"
def prettify_name(name: str) -> str:
    if name in pretty_map:
        return pretty_map[name]
    # Try to prettify OHE-style names: "col_value"
    if "_" in name:
        col, val = name.split("_", 1)
        col_lbl = pretty_map.get(col, col)
        return f"{col_lbl}: {val}"
    return name

pretty_feature_names = np.array([prettify_name(f) for f in feature_names])

# Optional: wrap long labels so they look nice in figures
pretty_feature_names = np.array(["\n".join(textwrap.wrap(x, width=28)) for x in pretty_feature_names])

# 3) Plot with publishable labels
shap.summary_plot(shap_vals, X_test_mat, feature_names=pretty_feature_names, max_display=20, show=True)
shap.summary_plot(shap_vals, X_test_mat, feature_names=pretty_feature_names, plot_type="bar", max_display=20, show=True)


In [ ]:
y = dat["plos"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

pipe.fit(X_train, y_train)
p = pipe.predict_proba(X_test)[:,1]
print("PLOS AUROC:", roc_auc_score(y_test, p))
print("PLOS AUPRC:", average_precision_score(y_test, p), "Test positives:", int(y_test.sum()))


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Use your ADRD cohort or full cohort
d = dat.copy()  # or df[df["adrd_any"]].copy()

d["los"]  = pd.to_numeric(d["los"], errors="coerce")
d["died"] = pd.to_numeric(d["died"], errors="coerce").fillna(0).astype(int)

d = d[(d["los"].notna()) & (d["los"] >= 0)].copy()

# LOS bins (edit as desired)
bins = [-0.1, 1, 3, 6, 9, 14, 21, 9999]
labels = ["0–1", "2–3", "4–6", "7–9", "10–14", "15–21", "22+"]

d["los_bin"] = pd.cut(d["los"], bins=bins, labels=labels)

tab = (d.groupby("los_bin", observed=True)
         .agg(n=("died","size"), deaths=("died","sum"), death_rate=("died","mean"))
         .reset_index())

tab["death_rate_pct"] = 100*tab["death_rate"]
print(tab)

plt.figure(figsize=(8,4))
plt.plot(tab["los_bin"].astype(str), tab["death_rate_pct"], marker="o")
plt.ylabel("In-hospital death rate (%)")
plt.xlabel("Length of stay (days)")
plt.xticks(rotation=0)
plt.grid(True, axis="y", alpha=0.3)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Your table (if it's already a DataFrame named tab, skip this)
tab = pd.DataFrame({
    "los_bin": ["0–1","2–3","4–6","7–9","10–14","15–21","22+"],
    "n":      [933, 3416, 3563, 1519, 1038, 521, 385],
    "deaths": [131, 137, 116, 70, 68, 50, 28],
})
tab["death_rate"] = tab["deaths"] / tab["n"]

def wilson_ci(k, n, z=1.96):
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = (z * np.sqrt((p*(1-p) + z**2/(4*n)) / n)) / denom
    return (center - half, center + half)

cis = tab.apply(lambda r: wilson_ci(r["deaths"], r["n"]), axis=1)
tab["ci_low"]  = [c[0] for c in cis]
tab["ci_high"] = [c[1] for c in cis]

# Convert to %
tab["death_rate_pct"] = 100 * tab["death_rate"]
tab["ci_low_pct"]     = 100 * tab["ci_low"]
tab["ci_high_pct"]    = 100 * tab["ci_high"]

# Error bars
yerr_low  = tab["death_rate_pct"] - tab["ci_low_pct"]
yerr_high = tab["ci_high_pct"] - tab["death_rate_pct"]

plt.figure(figsize=(9,4))
plt.errorbar(
    tab["los_bin"].astype(str),
    tab["death_rate_pct"],
    yerr=[yerr_low, yerr_high],
    fmt="o-",
    capsize=4
)
plt.xlabel("Length of stay (days)")
plt.ylabel("In-hospital death rate (%) with 95% CI")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

tab[["los_bin","n","deaths","death_rate_pct","ci_low_pct","ci_high_pct"]]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.arange(len(tab))

# Build x-axis labels with sample size
xticklabels = [f"{b}\n(n={n})" for b, n in zip(tab["los_bin"].astype(str), tab["n"].astype(int))]

y = tab["death_rate_pct"].values
yerr_low  = (tab["death_rate_pct"] - tab["ci_low_pct"]).values
yerr_high = (tab["ci_high_pct"] - tab["death_rate_pct"]).values

plt.figure(figsize=(9,4))
plt.errorbar(
    x, y,
    yerr=[yerr_low, yerr_high],
    fmt="o-",
    capsize=4
)
plt.xticks(x, xticklabels)
plt.xlabel("Length of stay (days)")
plt.ylabel("In-hospital death rate (%) with 95% CI")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
if "discwt" in d.columns:
    d["discwt"] = pd.to_numeric(d["discwt"], errors="coerce").fillna(1.0)
    wtab = (d.groupby("los_bin", observed=True)
              .apply(lambda g: pd.Series({
                  "n": len(g),
                  "weighted_death_rate": np.average(g["died"], weights=g["discwt"])
              }))
              .reset_index())
    wtab["weighted_death_rate_pct"] = 100*wtab["weighted_death_rate"]
    print(wtab)


In [ ]:
q = d.groupby("died")["los"].describe(percentiles=[0.25,0.5,0.75])
print(q)

plt.figure(figsize=(6,4))
d.boxplot(column="los", by="died")
plt.title("LOS by in-hospital death (0=survived, 1=died)")
plt.suptitle("")
plt.ylabel("LOS (days)")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrix

dd = d.copy()
dd["los"]  = pd.to_numeric(dd["los"], errors="coerce")
dd["died"] = pd.to_numeric(dd["died"], errors="coerce").fillna(0).astype(int)
dd = dd[(dd["los"].notna()) & (dd["los"] >= 0)].copy()

# Build spline basis
X_spline = dmatrix("bs(los, df=5, include_intercept=False)", data=dd, return_type="dataframe")

# Optional covariates (keep minimal)
covars = []
for c in ["age","female","sepsis","aki","i10_ndx","i10_npr"]:
    if c in dd.columns:
        dd[c] = pd.to_numeric(dd[c], errors="coerce")
        covars.append(c)

X = pd.concat([X_spline, dd[covars]], axis=1)
X = sm.add_constant(X, has_constant="add")

# --- critical: coerce everything to numeric float ---
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

y = dd["died"].astype(int)

# Fit using numpy arrays to avoid dtype=object issues
m = sm.GLM(y.to_numpy(dtype=float), X.to_numpy(dtype=float), family=sm.families.Binomial()).fit()
print(m.summary())


In [ ]:
import matplotlib.pyplot as plt
from patsy import dmatrix

los_grid = np.linspace(dd["los"].quantile(0.01), dd["los"].quantile(0.99), 200)
grid = pd.DataFrame({"los": los_grid})

Xg = dmatrix("bs(los, df=5, include_intercept=False)", data=grid, return_type="dataframe")
for c in covars:
    Xg[c] = dd[c].median(skipna=True)

Xg = sm.add_constant(Xg, has_constant="add")
Xg = Xg.apply(pd.to_numeric, errors="coerce").fillna(Xg.median(numeric_only=True))

pred = m.predict(Xg.to_numpy(dtype=float))

plt.figure(figsize=(7,4))
plt.plot(los_grid, 100*pred)
plt.xlabel("LOS (days)")
plt.ylabel("Predicted in-hospital death (%)")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Outcome
y = pd.to_numeric(dat["died"], errors="coerce").fillna(0).astype(int)

# Base predictors (mostly admission-time or baseline)
base_feats = [
    "age","female","pay1","zipinc_qrtl","pl_nchs","aweekend","elective","hcup_ed",
    "i10_ndx",  # dx count (available at coding, often reflects complexity)
    "sepsis","aki","chf","ckd","uti","stroke","pneumonia","delirium","copd",
    "ad_only"
]
base_feats = [c for c in base_feats if c in dat.columns]

# Post-admission proxies
post_feats = [c for c in ["i10_npr","los","totchg"] if c in dat.columns]

# Model A / Model B
features_A = [c for c in base_feats if c not in post_feats]          # admission-only
features_B = base_feats + post_feats                                 # full course

print("Model A features:", features_A)
print("Model B adds:", [c for c in features_B if c not in features_A])
print("N:", len(dat), "Deaths:", int(y.sum()))


In [ ]:
!pip -q install xgboost shap scikit-learn

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

def cv_xgb_shap(dat, y, features, n_splits=5, random_state=42):
    X = dat[features].copy()

    cat_cols = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith("string")]
    num_cols = [c for c in X.columns if c not in cat_cols]

    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_aurocs = []
    fold_top10 = []
    fold_importance = []

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        # imbalance handling
        pos = y_tr.sum()
        neg = len(y_tr) - pos
        spw = float(neg / pos) if pos > 0 else 1.0

        model = XGBClassifier(
            n_estimators=600, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, min_child_weight=1,
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", random_state=42,
            scale_pos_weight=spw
        )

        pipe = Pipeline([("pre", pre), ("model", model)])
        pipe.fit(X_tr, y_tr)

        p = pipe.predict_proba(X_te)[:, 1]
        auroc = roc_auc_score(y_te, p)
        fold_aurocs.append(auroc)

        # SHAP on validation fold
        X_te_mat = pipe.named_steps["pre"].transform(X_te)

        # feature names after OHE
        ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
        cat_names = ohe.get_feature_names_out(cat_cols) if len(cat_cols) else np.array([])
        feat_names = np.concatenate([np.array(num_cols), cat_names])

        explainer = shap.TreeExplainer(pipe.named_steps["model"])
        shap_vals = explainer.shap_values(X_te_mat)

        mean_abs = np.abs(shap_vals).mean(axis=0)
        imp = pd.Series(mean_abs, index=feat_names).sort_values(ascending=False)

        top10 = imp.head(10)
        fold_top10.append(top10.index.tolist())
        fold_importance.append(imp)

        print(f"Fold {fold}: AUROC={auroc:.3f} | Top features: {', '.join(top10.index[:5])} ...")

    # AUROC summary
    auroc_summary = {
        "mean": float(np.mean(fold_aurocs)),
        "sd": float(np.std(fold_aurocs, ddof=1)),
        "folds": fold_aurocs
    }

    # Stability: how often appears in top10 + average rank
    ranks = {}
    for k, top in enumerate(fold_top10):
        for r, f in enumerate(top, start=1):
            ranks.setdefault(f, []).append(r)

    stab = pd.DataFrame({
        "appear_top10_folds": {f: len(rs) for f, rs in ranks.items()},
        "avg_rank_when_in_top10": {f: float(np.mean(rs)) for f, rs in ranks.items()}
    }).sort_values(["appear_top10_folds","avg_rank_when_in_top10"], ascending=[False, True])

    return auroc_summary, fold_top10, stab

# Run Model A and Model B
y_series = y.reset_index(drop=True)
dat_use = dat.reset_index(drop=True)

print("\n=== Model A (admission-only) ===")
aucA, topA, stabA = cv_xgb_shap(dat_use, y_series, features_A)

print("\n=== Model B (full inpatient course) ===")
aucB, topB, stabB = cv_xgb_shap(dat_use, y_series, features_B)

print("\nAUROC Model A:", aucA)
print("AUROC Model B:", aucB)

print("\nStability table (Model A) top rows:")
display(stabA.head(20))

print("\nStability table (Model B) top rows:")
display(stabB.head(20))


In [ ]:
import statsmodels.api as sm
from patsy import dmatrix
import matplotlib.pyplot as plt

def fit_spline_curve(dd, covars=None, df_spline=5):
    dd = dd.copy()
    dd["los"]  = pd.to_numeric(dd["los"], errors="coerce")
    dd["died"] = pd.to_numeric(dd["died"], errors="coerce").fillna(0).astype(int)
    dd = dd[(dd["los"].notna()) & (dd["los"] >= 0)].copy()

    Xs = dmatrix(f"bs(los, df={df_spline}, include_intercept=False)", data=dd, return_type="dataframe")

    covars = covars or []
    for c in covars:
        if c in dd.columns:
            dd[c] = pd.to_numeric(dd[c], errors="coerce")

    X = pd.concat([Xs, dd[[c for c in covars if c in dd.columns]]], axis=1)
    X = sm.add_constant(X, has_constant="add")
    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    y = dd["died"].astype(int)
    m = sm.GLM(y.to_numpy(dtype=float), X.to_numpy(dtype=float), family=sm.families.Binomial()).fit()

    # prediction grid
    lo = dd["los"].quantile(0.01)
    hi = dd["los"].quantile(0.99)
    grid = pd.DataFrame({"los": np.linspace(lo, hi, 200)})

    Xg = dmatrix(f"bs(los, df={df_spline}, include_intercept=False)", data=grid, return_type="dataframe")
    for c in covars:
        if c in dd.columns:
            Xg[c] = dd[c].median(skipna=True)

    Xg = sm.add_constant(Xg, has_constant="add")
    Xg = Xg.apply(pd.to_numeric, errors="coerce").fillna(Xg.median(numeric_only=True))

    pred = m.predict(Xg.to_numpy(dtype=float))
    return grid["los"].values, pred

# Choose minimal covariates (optional)
covars = [c for c in ["age","female","sepsis","aki","i10_ndx"] if c in dat.columns]

dd0 = dat.copy()
dd1 = dat[dat["los"] > 1].copy()
dd2 = dat[dat["los"] > 2].copy()

x0, p0 = fit_spline_curve(dd0, covars=covars, df_spline=5)
x1, p1 = fit_spline_curve(dd1, covars=covars, df_spline=5)
x2, p2 = fit_spline_curve(dd2, covars=covars, df_spline=5)

plt.figure(figsize=(7,4))
plt.plot(x0, 100*p0, label="All LOS")
plt.plot(x1, 100*p1, label="Exclude LOS ≤ 1")
plt.plot(x2, 100*p2, label="Exclude LOS ≤ 2")
plt.xlabel("LOS (days)")
plt.ylabel("Predicted in-hospital death (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
def n_deaths(dd):
    return pd.Series({"N": len(dd), "Deaths": int(dd["died"].sum()), "DeathRate%": 100*dd["died"].mean()})

tbl = pd.DataFrame({
    "All": n_deaths(dat[dat["los"].notna()]),
    ">1 day": n_deaths(dat[(dat["los"] > 1) & dat["los"].notna()]),
    ">2 days": n_deaths(dat[(dat["los"] > 2) & dat["los"].notna()]),
}).T
tbl


In [ ]:
import numpy as np
import pandas as pd

# Use your AD-only dataset (replace if your AD cohort variable name differs)
# dat = df[df["ad_only"]].copy()

dat = dat.copy()
dat["los"] = pd.to_numeric(dat["los"], errors="coerce")

dx_cols = sorted([c for c in dat.columns if c.startswith("i10_dx")])

# Optional: exclude the principal diagnosis (often i10_dx1)
dx_cols_comorb = [c for c in dx_cols if c != "i10_dx1"]  # keep only secondary dx

def top_dx_prevalence(dsub, dx_cols_use, topn=20, code_level=3,
                      exclude_prefixes=("G30", "Z")):
    """
    Returns top ICD-10 codes by % of admissions with the code (deduplicated per admission).
    code_level=3 -> 3-character ICD-10 category (e.g., A41, N17).
    """
    dsub = dsub.copy()
    N = len(dsub)
    if N == 0:
        return pd.DataFrame(columns=["code","n_patients","pct_patients","N_group"])

    # long format (admission_id, code)
    long = (dsub[dx_cols_use]
            .astype("string")
            .apply(lambda s: s.str.upper().str.strip())
            .stack(future_stack=True))
    long = long[long.notna() & (long != "")]

    # exclude AD codes and optional Z-codes
    if exclude_prefixes:
        long = long[~long.str.startswith(tuple(exclude_prefixes), na=False)]

    # normalize code text
    # remove punctuation, keep alphanumerics
    code = long.str.replace(r"[^A-Z0-9]", "", regex=True)

    if code_level == 3:
        code = code.str.slice(0, 3)

    tmp = code.reset_index()
    tmp.columns = ["row_id", "dx_pos", "code"]
    tmp = tmp[tmp["code"].notna() & (tmp["code"] != "")]
    tmp = tmp.drop_duplicates(["row_id", "code"])  # prevalence per admission

    counts = tmp["code"].value_counts().head(topn)

    out = counts.rename("n_patients").to_frame()
    out["pct_patients"] = 100 * out["n_patients"] / N
    out = out.reset_index().rename(columns={"index": "code"})
    out["N_group"] = N
    return out

# Define LOS groups
short = dat[(dat["los"].notna()) & (dat["los"] <= 5)].copy()
long  = dat[(dat["los"].notna()) & (dat["los"] >= 25)].copy()

top_short = top_dx_prevalence(short, dx_cols_comorb, topn=25, code_level=3)
top_long  = top_dx_prevalence(long,  dx_cols_comorb, topn=25, code_level=3)

print("Short LOS (≤5) top comorbidities:")
display(top_short)

print("Long LOS (≥25) top comorbidities:")
display(top_long)


In [ ]:
comp = (top_short.merge(top_long, on="code", how="outer", suffixes=("_los_le5","_los_ge25"))
        .fillna(0))

comp["diff_pct"] = comp["pct_patients_los_ge25"] - comp["pct_patients_los_le5"]

print("Most common in short stays:")
display(comp.sort_values("pct_patients_los_le5", ascending=False).head(20))

print("Most common in long stays:")
display(comp.sort_values("pct_patients_los_ge25", ascending=False).head(20))

print("Most enriched in long stays (≥25 vs ≤5):")
display(comp.sort_values("diff_pct", ascending=False).head(20))


In [ ]:
import numpy as np
import pandas as pd

dat = dat.copy()
dat["los"] = pd.to_numeric(dat["los"], errors="coerce")
dx_cols = sorted([c for c in dat.columns if c.startswith("i10_dx")])

# Use secondary diagnoses only (exclude principal)
dx_cols_comorb = [c for c in dx_cols if c != "i10_dx1"]

# LOS groups
short = dat[(dat["los"].notna()) & (dat["los"] <= 5)].copy()
long  = dat[(dat["los"].notna()) & (dat["los"] >= 25)].copy()

# Exclude AD code and Z codes
EXCLUDE_PREFIXES_ALWAYS = ("G30", "Z")

# Exclude common acute/complication categories (prefix-level)
# (You can add/remove depending on what you want)
EXCLUDE_ACUTE_PREFIXES = (
    "A40","A41",      # sepsis
    "J12","J13","J14","J15","J16","J17","J18",  # pneumonia
    "J96",            # acute respiratory failure
    "R57",            # shock
    "N17",            # AKI
    "E86",            # volume depletion
    "R65",            # SIRS
    "I21","I22",      # acute MI
    "I26",            # pulmonary embolism
    "I60","I61","I62",# hemorrhagic stroke (optional: remove if you want strokes included)
    "T81","T82","T83","T84","T85","T88",  # procedural complications
)

def top_dx_prevalence_filtered(dsub, dx_cols_use, topn=25, code_level=3,
                               exclude_prefixes=EXCLUDE_PREFIXES_ALWAYS,
                               exclude_acute_prefixes=EXCLUDE_ACUTE_PREFIXES,
                               keep_stroke=True):
    N = len(dsub)
    if N == 0:
        return pd.DataFrame(columns=["code","n_adm","pct_adm","N_group"])

    long_s = (dsub[dx_cols_use]
              .astype("string")
              .apply(lambda s: s.str.upper().str.strip())
              .stack(future_stack=True))
    long_s = long_s[long_s.notna() & (long_s != "")]

    # normalize
    code = long_s.str.replace(r"[^A-Z0-9]", "", regex=True)

    # optional 3-char category
    if code_level == 3:
        code3 = code.str.slice(0, 3)
    else:
        code3 = code

    # Always exclude AD and Z-codes
    code3 = code3[~code3.str.startswith(tuple(exclude_prefixes), na=False)]

    # Acute exclusions
    acute = exclude_acute_prefixes
    if keep_stroke:
        # keep ischemic/hemorrhagic stroke categories if you want them considered comorbid
        acute = tuple([p for p in acute if p not in ("I60","I61","I62","I63")])

    code3 = code3[~code3.str.startswith(acute, na=False)]

    tmp = code3.reset_index()
    tmp.columns = ["row_id", "dx_pos", "code"]
    tmp = tmp[tmp["code"].notna() & (tmp["code"] != "")]
    tmp = tmp.drop_duplicates(["row_id", "code"])  # prevalence per admission

    counts = tmp["code"].value_counts().head(topn)

    out = counts.rename("n_adm").to_frame()
    out["pct_adm"] = 100 * out["n_adm"] / N
    out = out.reset_index().rename(columns={"index":"code"})
    out["N_group"] = N
    return out

top_short_chronic = top_dx_prevalence_filtered(short, dx_cols_comorb, topn=30, keep_stroke=True)
top_long_chronic  = top_dx_prevalence_filtered(long,  dx_cols_comorb, topn=30, keep_stroke=True)

print("Short LOS (≤5) chronic-focused top codes:")
display(top_short_chronic)

print("Long LOS (≥25) chronic-focused top codes:")
display(top_long_chronic)


In [ ]:
import numpy as np
import pandas as pd

# AD-only cohort (adjust if your flag name differs)
d = dat.copy()  # should already be AD-only. If not: d = df[df["ad_only"]==1].copy()

# Outcome
d["died"] = pd.to_numeric(d["died"], errors="coerce").fillna(0).astype(int)

# Optional: ensure key continuous vars are numeric if present
for c in ["age","los","totchg","i10_ndx","i10_npr"]:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

print("N:", len(d), "Deaths:", int(d["died"].sum()), "Death%:", 100*d["died"].mean())


In [ ]:
# Base predictors (edit to match what you used)
base_feats = [
    "age","female","pay1","zipinc_qrtl","pl_nchs","aweekend","elective","hcup_ed",
    "i10_ndx", "sepsis","aki","uti","chf","ckd","stroke","pneumonia","delirium","copd"
]
base_feats = [c for c in base_feats if c in d.columns]

# Course-aware additions (only if present)
course_feats = [c for c in ["los","totchg","i10_npr"] if c in d.columns]

features_A = [c for c in base_feats if c not in course_feats]
features_B = base_feats + course_feats

print("Model A predictors:", features_A)
print("Model B adds:", [c for c in features_B if c not in features_A])


In [ ]:
import statsmodels.api as sm

def fit_logit_glm(df, y_col, x_cols, weight_col=None, cluster_col=None):
    df = df.copy()

    # y
    y = pd.to_numeric(df[y_col], errors="coerce").fillna(0).astype(int)

    # X: dummies for categoricals, numeric for everything else
    X = df[x_cols].copy()

    # Convert booleans to int early (avoids dtype issues)
    for c in X.columns:
        if X[c].dtype == bool:
            X[c] = X[c].astype(int)

    # One-hot encode non-numeric columns
    X = pd.get_dummies(X, drop_first=True, dummy_na=True)

    # Force numeric matrix
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    # Add intercept
    X = sm.add_constant(X, has_constant="add")

    # Weights
    if weight_col and weight_col in df.columns:
        w = pd.to_numeric(df[weight_col], errors="coerce").fillna(1.0).to_numpy(dtype=float)
    else:
        w = None

    # Fit GLM (Binomial)
    model = sm.GLM(y.to_numpy(dtype=float), X.to_numpy(dtype=float),
                   family=sm.families.Binomial(),
                   freq_weights=w)

    if cluster_col and cluster_col in df.columns:
        groups = df[cluster_col].to_numpy()
        res = model.fit(cov_type="cluster", cov_kwds={"groups": groups})
    else:
        res = model.fit(cov_type="HC1")

    return res, X.columns


In [ ]:
weight_col = "discwt" if "discwt" in d.columns else None
cluster_col = "hosp_nrd" if "hosp_nrd" in d.columns else None

print("Using weights:", weight_col, "| Cluster SE by:", cluster_col)

resA, colsA = fit_logit_glm(d, "died", features_A, weight_col=weight_col, cluster_col=cluster_col)
resB, colsB = fit_logit_glm(d, "died", features_B, weight_col=weight_col, cluster_col=cluster_col)

print(resA.summary())
print(resB.summary())


In [ ]:
def or_table(res, colnames, top=25):
    b = pd.Series(res.params, index=colnames)
    se = pd.Series(res.bse, index=colnames)
    p = pd.Series(res.pvalues, index=colnames)

    out = pd.DataFrame({
        "OR": np.exp(b),
        "CI_low": np.exp(b - 1.96*se),
        "CI_high": np.exp(b + 1.96*se),
        "p": p
    })
    out = out.drop(index=["const"], errors="ignore")
    out = out.sort_values("p")
    return out.head(top), out

topA, fullA = or_table(resA, colsA, top=30)
topB, fullB = or_table(resB, colsB, top=30)

print("Top predictors (Model A):")
display(topA)

print("Top predictors (Model B):")
display(topB)


In [ ]:
from sklearn.metrics import roc_auc_score

def glm_auc(res, Xcols, df, y_col):
    y = pd.to_numeric(df[y_col], errors="coerce").fillna(0).astype(int).to_numpy()
    # rebuild X with same columns used in the model
    X = df.copy()
    return y

# Easiest is to compute AUROC using in-sample predicted probabilities:
pA = resA.fittedvalues
pB = resB.fittedvalues
y = d["died"].astype(int).to_numpy()

print("In-sample AUROC (Model A):", roc_auc_score(y, pA))
print("In-sample AUROC (Model B):", roc_auc_score(y, pB))


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

def fit_logit_safe(df, y_col, x_cols):
    df = df.copy()

    y = pd.to_numeric(df[y_col], errors="coerce").fillna(0).astype(int).to_numpy()

    X = df[x_cols].copy()

    # Convert booleans to int
    for c in X.columns:
        if X[c].dtype == bool:
            X[c] = X[c].astype(int)

    # One-hot encode categoricals (NO dummy_na)
    X = pd.get_dummies(X, drop_first=True, dummy_na=False)

    # Force numeric; convert Inf to NaN
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)

    # Fill NaN with median; if median is NaN (all missing), fill with 0
    med = X.median(numeric_only=True)
    X = X.fillna(med)
    X = X.fillna(0)

    # Drop columns that are constant (helps stability)
    nunique = X.nunique(dropna=False)
    X = X.loc[:, nunique > 1]

    # Add intercept
    X = sm.add_constant(X, has_constant="add")

    # Final safety check
    arr = X.to_numpy(dtype=float)
    if not np.isfinite(arr).all():
        bad_cols = X.columns[~np.isfinite(arr).all(axis=0)]
        print("Non-finite columns:", list(bad_cols))
        raise ValueError("Design matrix still has NaN/Inf.")

    res = sm.GLM(y, arr, family=sm.families.Binomial()).fit(cov_type="HC1")
    return res, X.columns


In [ ]:
from patsy import dmatrix

d = dat.copy()
d["died"] = pd.to_numeric(d["died"], errors="coerce").fillna(0).astype(int)

for c in ["age","i10_ndx","i10_npr","los","totchg"]:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

# Scale charges
if "totchg" in d.columns:
    d["totchg_10k"] = d["totchg"] / 10000.0

# Impute categorical missings to mode (prevents *_nan separation)
cat_cols = [c for c in ["pay1","pl_nchs","zipinc_qrtl","elective","hcup_ed","aweekend"] if c in d.columns]
for c in cat_cols:
    m = d[c].mode(dropna=True)
    if len(m) > 0:
        d[c] = d[c].fillna(m.iloc[0])

features_A = [c for c in ["age","female","pay1","zipinc_qrtl","pl_nchs","aweekend","elective","hcup_ed",
                          "i10_ndx","sepsis","aki","stroke","chf","ckd","uti","pneumonia","delirium","copd"]
              if c in d.columns]

# Model A fit (no LOS required)
resA, colsA = fit_logit_safe(d.dropna(subset=["died"]), "died", features_A)

# Model B: require LOS for spline
dB = d.dropna(subset=["died","los"]).copy()

# Add LOS spline basis
los_spline = dmatrix("bs(los, df=5, include_intercept=False)", data=dB, return_type="dataframe")
los_spline.columns = [f"los_s{i}" for i in range(los_spline.shape[1])]
dB = pd.concat([dB, los_spline], axis=1)

features_B = features_A.copy()
for c in ["i10_npr","totchg_10k"]:
    if c in dB.columns:
        features_B.append(c)
features_B += list(los_spline.columns)

resB, colsB = fit_logit_safe(dB, "died", features_B)


In [ ]:
def or_table(res, colnames, top=30):
    b = pd.Series(res.params, index=colnames)
    se = pd.Series(res.bse, index=colnames)
    p = pd.Series(res.pvalues, index=colnames)
    out = pd.DataFrame({
        "OR": np.exp(b),
        "CI_low": np.exp(b - 1.96*se),
        "CI_high": np.exp(b + 1.96*se),
        "p": p
    }).drop(index=["const"], errors="ignore").sort_values("p")
    return out.head(top), out

topA, fullA = or_table(resA, colsA)
topB, fullB = or_table(resB, colsB)

display(topA)
display(topB)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrix

d = dat.copy()
d["died"] = pd.to_numeric(d["died"], errors="coerce").fillna(0).astype(int)

# Scale charges (if available)
if "totchg" in d.columns:
    d["totchg_10k"] = pd.to_numeric(d["totchg"], errors="coerce") / 10000.0

# Ensure numerics
for c in ["age","i10_ndx","i10_npr","los"]:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

# Impute categorical missings to mode (prevents dummy_na separation)
cat_cols = [c for c in ["pay1","pl_nchs","zipinc_qrtl","elective","hcup_ed","aweekend"] if c in d.columns]
for c in cat_cols:
    m = d[c].mode(dropna=True)
    if len(m) > 0:
        d[c] = d[c].fillna(m.iloc[0])

def fit_logit(df, y_col, x_cols):
    X = df[x_cols].copy()

    # Convert booleans to int
    for c in X.columns:
        if X[c].dtype == bool:
            X[c] = X[c].astype(int)

    # one-hot encode categoricals WITHOUT dummy_na
    X = pd.get_dummies(X, drop_first=True, dummy_na=False)
    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)

    # Fill NaN with median; if median is NaN (all missing), fill with 0
    med = X.median(numeric_only=True)
    X = X.fillna(med)
    X = X.fillna(0)

    # Drop columns that are constant (helps stability)
    nunique = X.nunique(dropna=False)
    X = X.loc[:, nunique > 1]

    X = sm.add_constant(X, has_constant="add")
    y = df[y_col].astype(int).to_numpy()
    res = sm.GLM(y, X.to_numpy(dtype=float), family=sm.families.Binomial()).fit(cov_type="HC1")
    return res, X.columns

# Model A predictors (edit as needed)
features_A = [c for c in ["age","female","pay1","zipinc_qrtl","pl_nchs","aweekend","elective","hcup_ed",
                          "i10_ndx","sepsis","aki","stroke","chf","ckd","uti","pneumonia","delirium","copd"]
              if c in d.columns]

# Model B predictors (course-aware)
features_B = features_A.copy()
for c in ["i10_npr","totchg_10k"]:
    if c in d.columns:
        features_B.append(c)

# OPTIONAL: LOS as spline (recommended) instead of linear
if "los" in d.columns:
    los_spline = dmatrix("bs(los, df=5, include_intercept=False)", data=d, return_type="dataframe")
    los_spline.columns = [f"los_s{i}" for i in range(los_spline.shape[1])]
    d = pd.concat([d, los_spline], axis=1)
    features_B = [c for c in features_B if c != "los"] + list(los_spline.columns)

resA, colsA = fit_logit(d.dropna(subset=["died"]), "died", features_A)
resB, colsB = fit_logit(d.dropna(subset=["died"]), "died", features_B)

def or_table(res, colnames):
    b = pd.Series(res.params, index=colnames)
    se = pd.Series(res.bse, index=colnames)
    out = pd.DataFrame({
        "OR": np.exp(b),
        "CI_low": np.exp(b - 1.96*se),
        "CI_high": np.exp(b + 1.96*se),
        "p": res.pvalues
    }).drop(index=["const"], errors="ignore").sort_values("p")
    return out

tabA = or_table(resA, colsA)
tabB = or_table(resB, colsB)

display(tabA.head(25))
display(tabB.head(25))


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrix

# ---------------------------
# Helpers
# ---------------------------
def set_reference_category(s, ref):
    """Make ref the first category so get_dummies(drop_first=True) uses it as reference."""
    s = s.astype("Int64")
    cats = sorted([c for c in s.dropna().unique().tolist() if pd.notna(c)])
    if ref in cats:
        cats = [ref] + [c for c in cats if c != ref]
    return s.astype(pd.CategoricalDtype(categories=cats))

def fit_logit_safe(df, y_col, x_cols, weight_col=None, cluster_col=None):
    df = df.copy()

    y = pd.to_numeric(df[y_col], errors="coerce").fillna(0).astype(int).to_numpy()

    X = df[x_cols].copy()
    for c in X.columns:
        if X[c].dtype == bool:
            X[c] = X[c].astype(int)

    X = pd.get_dummies(X, drop_first=True, dummy_na=False)
    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

    med = X.median(numeric_only=True)
    X = X.fillna(med).fillna(0)

    nunique = X.nunique(dropna=False)
    X = X.loc[:, nunique > 1]

    X = sm.add_constant(X, has_constant="add")

    w = None
    if weight_col and weight_col in df.columns:
        w = pd.to_numeric(df[weight_col], errors="coerce").fillna(1.0).to_numpy(dtype=float)

    model = sm.GLM(y, X.to_numpy(dtype=float), family=sm.families.Binomial(), freq_weights=w)

    if cluster_col and cluster_col in df.columns:
        groups = df[cluster_col].to_numpy()
        res = model.fit(cov_type="cluster", cov_kwds={"groups": groups})
    else:
        res = model.fit(cov_type="HC1")

    return res, X.columns

def or_table(res, colnames):
    b = pd.Series(res.params, index=colnames)
    se = pd.Series(res.bse, index=colnames)
    p = pd.Series(res.pvalues, index=colnames)
    out = pd.DataFrame({
        "OR": np.exp(b),
        "CI_low": np.exp(b - 1.96*se),
        "CI_high": np.exp(b + 1.96*se),
        "p": p
    })
    out = out.drop(index=["const"], errors="ignore")
    return out

def fmt_or(orv, lo, hi):
    return f"{orv:.2f} ({lo:.2f}–{hi:.2f})"

def fmt_p(p):
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

def make_jad_table(tabA, tabB, label_map=None, drop_like=("los_s",), keep_order=None):
    """Build a clean Table 2 combining Model A and Model B."""
    label_map = label_map or {}

    # remove spline terms from display
    def drop_terms(tab):
        idx = tab.index.astype(str)
        keep = np.ones(len(idx), dtype=bool)
        for pat in drop_like:
            keep &= ~idx.str.startswith(pat)
        return tab.loc[idx[keep]].copy()

    tA = drop_terms(tabA)
    tB = drop_terms(tabB)

    all_idx = sorted(set(tA.index).union(set(tB.index)))
    out = []
    for k in all_idx:
        a = tA.loc[k] if k in tA.index else None
        b = tB.loc[k] if k in tB.index else None
        out.append({
            "Predictor": label_map.get(k, k),
            "Model A OR (95% CI)": "" if a is None else fmt_or(a.OR, a.CI_low, a.CI_high),
            "p (A)": "" if a is None else fmt_p(a.p),
            "Model B OR (95% CI)": "" if b is None else fmt_or(b.OR, b.CI_low, b.CI_high),
            "p (B)": "" if b is None else fmt_p(b.p),
        })

    df_out = pd.DataFrame(out)

    if keep_order:
        # keep_order can be a list of predictors (raw names) to appear first
        order = [label_map.get(k, k) for k in keep_order]
        df_out["__order"] = df_out["Predictor"].apply(lambda x: order.index(x) if x in order else 10**9)
        df_out = df_out.sort_values(["__order", "Predictor"]).drop(columns="__order")

    return df_out

# ---------------------------
# YOUR DATA
# ---------------------------
d = dat.copy()  # should be AD-only already
d["died"] = pd.to_numeric(d["died"], errors="coerce").fillna(0).astype(int)

# numeric vars
for c in ["age","i10_ndx","i10_npr","los","totchg"]:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

# scale charges
if "totchg" in d.columns:
    d["totchg_10k"] = d["totchg"] / 10000.0

# force categorical variables with common NRD references
# PAY1: ref=1 (Medicare); ZIPINC_QRTL: ref=1 (lowest); PL_NCHS: ref=1
if "pay1" in d.columns:
    d["pay1"] = set_reference_category(d["pay1"], ref=1)
if "zipinc_qrtl" in d.columns:
    d["zipinc_qrtl"] = set_reference_category(d["zipinc_qrtl"], ref=1)
if "pl_nchs" in d.columns:
    d["pl_nchs"] = set_reference_category(d["pl_nchs"], ref=1)

# binary context vars: set reference as 0 if present
for c in ["elective","hcup_ed","aweekend","female"]:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0).astype(int)
        d[c] = set_reference_category(d[c], ref=0)

# Optional: weights / cluster
weight_col  = "discwt" if "discwt" in d.columns else None
cluster_col = "hosp_nrd" if "hosp_nrd" in d.columns else None
print("Weights:", weight_col, "| Cluster:", cluster_col)

# ---------------------------
# FEATURES (edit only if needed)
# ---------------------------
features_A = [c for c in [
    "age","female","pay1","zipinc_qrtl","pl_nchs","aweekend","elective","hcup_ed",
    "i10_ndx","sepsis","aki","stroke","chf","ckd","uti","pneumonia","delirium","copd"
] if c in d.columns]

# Model A
resA, colsA = fit_logit_safe(d.dropna(subset=["died"]), "died", features_A, weight_col=weight_col, cluster_col=cluster_col)
tabA = or_table(resA, colsA)

# Model B requires LOS (for spline)
dB = d.dropna(subset=["died","los"]).copy()

# LOS spline
los_spline = dmatrix("bs(los, df=5, include_intercept=False)", data=dB, return_type="dataframe")
los_spline.columns = [f"los_s{i}" for i in range(los_spline.shape[1])]
dB = pd.concat([dB, los_spline], axis=1)

features_B = features_A.copy()
for c in ["i10_npr","totchg_10k"]:
    if c in dB.columns:
        features_B.append(c)
features_B += list(los_spline.columns)

resB, colsB = fit_logit_safe(dB, "died", features_B, weight_col=weight_col, cluster_col=cluster_col)
tabB = or_table(resB, colsB)

# Joint test for LOS spline terms (report this p-value in the table footnote)
los_idx = [i for i,c in enumerate(colsB) if str(c).startswith("los_s")]
R = np.zeros((len(los_idx), len(colsB)))
for r,j in enumerate(los_idx):
    R[r,j] = 1
los_joint_p = float(resB.wald_test(R).pvalue)
print("Joint LOS spline p-value:", los_joint_p)

# Friendly labels (optional)
label_map = {
    "age":"Age (per year)",
    "female_1":"Female sex (ref: male)",
    "i10_ndx":"Number of diagnoses (I10_NDX)",
    "i10_npr":"Number of procedures (I10_NPR)",
    "totchg_10k":"Total charges (per $10,000)",
    "hcup_ed_1":"ED involvement (ref: no ED)",
    "aweekend_1":"Weekend admission (ref: weekday)",
    "elective_1":"Elective admission (ref: non-elective)",
    "sepsis":"Sepsis",
    "aki":"Acute kidney injury",
    "stroke":"Stroke",
    "chf":"Congestive heart failure",
    "ckd":"Chronic kidney disease",
    "uti":"Urinary tract infection",
    "pneumonia":"Pneumonia",
    "delirium":"Delirium",
    "copd":"COPD",
}

# Build Table 2
keep_order = ["sepsis","aki","stroke","i10_ndx","age","chf","uti","ckd","female_1","hcup_ed_1","aweekend_1","i10_npr","totchg_10k"]
table2 = make_jad_table(tabA, tabB, label_map=label_map, keep_order=keep_order)

display(table2)

print("\nSuggested footnote:")
print(f"Model B included length of stay (LOS) modeled with restricted cubic splines (df=5); spline terms are not shown. "
      f"LOS spline terms were jointly significant (Wald p={los_joint_p:.3g}). "
      "PAY1, ZIP income quartile, PL_NCHS, and admission context variables were modeled as categorical with stated references.")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind

# =========================
# CONFIG
# =========================
df = dat.copy()  # <-- your AD-only analytic dataset
wcol = "discwt"  # NRD discharge weight
ycol = "died"    # in-hospital death (0/1)

# Variables for Table 1
cont_mean_sd = [c for c in ["age", "i10_ndx"] if c in df.columns]
cont_med_iqr = [c for c in ["los", "i10_npr", "totchg"] if c in df.columns]

cat_vars = [c for c in [
    "female", "pay1", "zipinc_qrtl", "pl_nchs", "aweekend", "elective", "hcup_ed",
    "sepsis", "aki", "stroke", "chf", "ckd", "uti", "pneumonia", "delirium", "copd"
] if c in df.columns]

# Optional labels
pay1_labels = {1:"Medicare", 2:"Medicaid", 3:"Private", 4:"Self-pay", 5:"No charge", 6:"Other"}
zip_labels  = {1:"Q1 (lowest)", 2:"Q2", 3:"Q3", 4:"Q4 (highest)"}
pl_labels   = {1:"Central metro", 2:"Fringe metro", 3:"Medium metro", 4:"Small metro", 5:"Micropolitan", 6:"Noncore"}

# =========================
# HELPERS
# =========================
def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    m = np.average(x, weights=w)
    return m

def weighted_sd(x, w):
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    m = np.average(x, weights=w)
    v = np.average((x - m)**2, weights=w)
    return np.sqrt(v)

def weighted_quantile(values, quantiles, sample_weight):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles)
    sample_weight = np.asarray(sample_weight, dtype=float)

    mask = np.isfinite(values) & np.isfinite(sample_weight)
    values, sample_weight = values[mask], sample_weight[mask]
    if len(values) == 0:
        return [np.nan for _ in quantiles]

    sorter = np.argsort(values)
    values, sample_weight = values[sorter], sample_weight[sorter]
    cum_w = np.cumsum(sample_weight)
    if cum_w[-1] == 0:
        return [np.nan for _ in quantiles]
    cum_w = cum_w / cum_w[-1]
    return np.interp(quantiles, cum_w, values)

def fmt_mean_sd(m, sd):
    return f"{m:.2f} ({sd:.2f})"

def fmt_med_iqr(q50, q25, q75):
    return f"{q50:.1f} ({q25:.1f}–{q75:.1f})"

def fmt_n_pct(n, pct):
    return f"{n:,.0f} ({pct:.1f})"

def weighted_cat_table(dsub, var, wcol):
    x = dsub[var]
    w = dsub[wcol]
    tmp = pd.DataFrame({var: x, wcol: w}).dropna(subset=[var, wcol])
    if tmp.empty:
        return pd.Series(dtype=float), 0.0
    total_w = tmp[wcol].sum()
    wcounts = tmp.groupby(var)[wcol].sum().sort_index()
    return wcounts, total_w

def pvalue_weighted_chi2(var):
    # Approximate weighted chi-square using weighted counts (not Rao-Scott)
    d = df[[var, ycol, wcol]].dropna()
    if d.empty:
        return np.nan
    tab = d.pivot_table(index=var, columns=ycol, values=wcol, aggfunc="sum", fill_value=0)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan
    chi2, p, _, _ = chi2_contingency(tab.values)
    return p

def pvalue_unweighted_t(var):
    d = df[[var, ycol]].dropna()
    if d.empty:
        return np.nan
    a = d.loc[d[ycol]==0, var].values
    b = d.loc[d[ycol]==1, var].values
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return ttest_ind(a, b, equal_var=False).pvalue

# =========================
# CLEAN CORE COLUMNS
# =========================
df[ycol] = _to_num(df[ycol]).fillna(0).astype(int)
df[wcol] = _to_num(df[wcol]).fillna(1.0)

for c in cont_mean_sd + cont_med_iqr:
    df[c] = _to_num(df[c])

for c in cat_vars:
    # Keep categories as-is; ensure numeric for binaries
    if df[c].dropna().isin([0,1]).all():
        df[c] = _to_num(df[c]).fillna(0).astype(int)

# Group subsets
df0 = df[df[ycol]==0].copy()
df1 = df[df[ycol]==1].copy()

W_all = df[wcol].sum()
W_0   = df0[wcol].sum()
W_1   = df1[wcol].sum()

# =========================
# BUILD TABLE 1
# =========================
rows = []

# Sample size row
rows.append({
    "Variable": "Weighted N (sum of discwt)",
    "Overall": f"{W_all:,.0f}",
    "Survived": f"{W_0:,.0f}",
    "Died": f"{W_1:,.0f}",
    "p-value": ""
})
rows.append({
    "Variable": "Unweighted N",
    "Overall": f"{len(df):,}",
    "Survived": f"{len(df0):,}",
    "Died": f"{len(df1):,}",
    "p-value": ""
})

# Continuous: mean (SD)
for v in cont_mean_sd:
    d_all = df[[v, wcol]].dropna()
    d_0   = df0[[v, wcol]].dropna()
    d_1   = df1[[v, wcol]].dropna()

    m_all, sd_all = weighted_mean(d_all[v], d_all[wcol]), weighted_sd(d_all[v], d_all[wcol])
    m_0, sd_0     = weighted_mean(d_0[v], d_0[wcol]), weighted_sd(d_0[v], d_0[wcol])
    m_1, sd_1     = weighted_mean(d_1[v], d_1[wcol]), weighted_sd(d_1[v], d_1[wcol])

    rows.append({
        "Variable": v,
        "Overall": fmt_mean_sd(m_all, sd_all),
        "Survived": fmt_mean_sd(m_0, sd_0),
        "Died": fmt_mean_sd(m_1, sd_1),
        "p-value": f"{pvalue_unweighted_t(v):.3g}" if np.isfinite(pvalue_unweighted_t(v)) else ""
    })

# Continuous: median (IQR)
for v in cont_med_iqr:
    d_all = df[[v, wcol]].dropna()
    d_0   = df0[[v, wcol]].dropna()
    d_1   = df1[[v, wcol]].dropna()

    q_all = weighted_quantile(d_all[v], [0.25, 0.50, 0.75], d_all[wcol])
    q_0   = weighted_quantile(d_0[v],   [0.25, 0.50, 0.75], d_0[wcol])
    q_1   = weighted_quantile(d_1[v],   [0.25, 0.50, 0.75], d_1[wcol])

    rows.append({
        "Variable": v,
        "Overall": fmt_med_iqr(q_all[1], q_all[0], q_all[2]),
        "Survived": fmt_med_iqr(q_0[1], q_0[0], q_0[2]),
        "Died": fmt_med_iqr(q_1[1], q_1[0], q_1[2]),
        "p-value": ""
    })

# Categorical variables (weighted N and %)
def label_for(var, level):
    if var == "pay1":
        return pay1_labels.get(level, str(level))
    if var == "zipinc_qrtl":
        return zip_labels.get(level, str(level))
    if var == "pl_nchs":
        return pl_labels.get(level, str(level))
    return str(level)

for v in cat_vars:
    # header row
    rows.append({"Variable": v, "Overall":"", "Survived":"", "Died":"", "p-value": ""})

    w_all, tot_all = weighted_cat_table(df, v, wcol)
    w_0, tot_0     = weighted_cat_table(df0, v, wcol)
    w_1, tot_1     = weighted_cat_table(df1, v, wcol)

    levels = sorted(set(w_all.index.tolist()) | set(w_0.index.tolist()) | set(w_1.index.tolist()))
    pcat = pvalue_weighted_chi2(v)

    for lv in levels:
        n_all = w_all.get(lv, 0.0); pct_all = (n_all/tot_all*100) if tot_all>0 else np.nan
        n_0   = w_0.get(lv, 0.0);   pct_0   = (n_0/tot_0*100) if tot_0>0 else np.nan
        n_1   = w_1.get(lv, 0.0);   pct_1   = (n_1/tot_1*100) if tot_1>0 else np.nan

        rows.append({
            "Variable": f"  {label_for(v, lv)}",
            "Overall": fmt_n_pct(n_all, pct_all) if np.isfinite(pct_all) else "",
            "Survived": fmt_n_pct(n_0, pct_0) if np.isfinite(pct_0) else "",
            "Died": fmt_n_pct(n_1, pct_1) if np.isfinite(pct_1) else "",
            "p-value": f"{pcat:.3g}" if np.isfinite(pcat) else ""
        })

table1 = pd.DataFrame(rows)

# Optional: rename some common variables for presentation
rename_map = {
    "age":"Age, years",
    "female":"Sex",
    "i10_ndx":"Number of diagnoses (I10_NDX)",
    "i10_npr":"Number of procedures (I10_NPR)",
    "los":"Length of stay, days",
    "totchg":"Total charges",
    "pay1":"Primary payer",
    "zipinc_qrtl":"ZIP income quartile",
    "pl_nchs":"Urban–rural category (PL_NCHS)",
    "aweekend":"Weekend admission",
    "elective":"Elective admission",
    "hcup_ed":"ED variable (HCUP_ED)",
    "sepsis":"Sepsis",
    "aki":"Acute kidney injury",
    "stroke":"Stroke",
    "chf":"Congestive heart failure",
    "ckd":"Chronic kidney disease",
    "uti":"Urinary tract infection",
    "pneumonia":"Pneumonia",
    "delirium":"Delirium",
    "copd":"COPD"
}
table1["Variable"] = table1["Variable"].replace(rename_map)

display(table1)

# Save
table1.to_csv("Table1_weighted_descriptives.csv", index=False)
print("Saved: Table1_weighted_descriptives.csv")


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind

# =========================
# INPUTS
# =========================
df = dat.copy()              # your AD-only cohort
wcol = "discwt"              # NRD discharge weight
ycol = "died"                # 0/1 in-hospital death

# Pretty labels for selected categorical variables (edit if you want)
pay1_labels = {1:"Medicare", 2:"Medicaid", 3:"Private", 4:"Self-pay", 5:"No charge", 6:"Other"}
zip_labels  = {1:"Q1 (lowest)", 2:"Q2", 3:"Q3", 4:"Q4 (highest)"}
pl_labels   = {1:"Central metro", 2:"Fringe metro", 3:"Medium metro", 4:"Small metro", 5:"Micropolitan", 6:"Noncore"}

# Optional: if you know HCUP_ED meanings for your file, map them here; otherwise it will show raw levels
hcup_ed_labels = {0:"0", 1:"1", 2:"2", 4:"4"}

rename_map = {
    "age":"Age, years",
    "female":"Sex (female)",
    "pay1":"Primary payer",
    "zipinc_qrtl":"ZIP income quartile",
    "pl_nchs":"Urban–rural category (PL_NCHS)",
    "aweekend":"Weekend admission",
    "elective":"Elective admission",
    "hcup_ed":"ED variable (HCUP_ED)",
    "i10_ndx":"Number of diagnoses (I10_NDX)",
    "i10_npr":"Number of procedures (I10_NPR)",
    "los":"Length of stay, days",
    "totchg":"Total charges",
    "sepsis":"Sepsis",
    "aki":"Acute kidney injury",
    "stroke":"Stroke",
    "chf":"Congestive heart failure",
    "ckd":"Chronic kidney disease",
    "uti":"Urinary tract infection",
    "pneumonia":"Pneumonia",
    "delirium":"Delirium",
    "copd":"COPD"
}

# =========================
# HELPERS
# =========================
def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    return np.average(x, weights=w)

def weighted_sd(x, w):
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    m = np.average(x, weights=w)
    v = np.average((x - m)**2, weights=w)
    return np.sqrt(v)

def weighted_quantile(values, quantiles, sample_weight):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles)
    sample_weight = np.asarray(sample_weight, dtype=float)
    mask = np.isfinite(values) & np.isfinite(sample_weight)
    values, sample_weight = values[mask], sample_weight[mask]
    if len(values) == 0 or sample_weight.sum() == 0:
        return [np.nan for _ in quantiles]
    sorter = np.argsort(values)
    values, sample_weight = values[sorter], sample_weight[sorter]
    cum_w = np.cumsum(sample_weight)
    cum_w = cum_w / cum_w[-1]
    return np.interp(quantiles, cum_w, values)

def fmt_mean_sd(m, sd):
    return f"{m:.2f} ({sd:.2f})"

def fmt_med_iqr(q50, q25, q75):
    return f"{q50:.1f} ({q25:.1f}–{q75:.1f})"

def fmt_n_pct(n, pct):
    return f"{n:,.0f} ({pct:.1f})"

def p_unweighted_t(d, v):
    d2 = d[[v, ycol]].dropna()
    if d2.empty: return np.nan
    a = d2.loc[d2[ycol]==0, v].values
    b = d2.loc[d2[ycol]==1, v].values
    if len(a) < 2 or len(b) < 2: return np.nan
    return ttest_ind(a, b, equal_var=False).pvalue

def p_unweighted_chi2(d, v):
    d2 = d[[v, ycol]].dropna()
    if d2.empty: return np.nan
    tab = pd.crosstab(d2[v], d2[ycol])
    if tab.shape[0] < 2 or tab.shape[1] < 2: return np.nan
    chi2, p, _, _ = chi2_contingency(tab.values)
    return p

def weighted_cat_counts(dsub, var):
    tmp = dsub[[var, wcol]].dropna()
    if tmp.empty:
        return pd.Series(dtype=float), 0.0
    total_w = tmp[wcol].sum()
    wcounts = tmp.groupby(var)[wcol].sum().sort_index()
    return wcounts, total_w

def label_level(var, level):
    if var == "pay1": return pay1_labels.get(level, str(level))
    if var == "zipinc_qrtl": return zip_labels.get(level, str(level))
    if var == "pl_nchs": return pl_labels.get(level, str(level))
    if var == "hcup_ed": return hcup_ed_labels.get(level, str(level))
    return str(level)

def build_table1(d, include_course_vars=False):
    d = d.copy()

    # core cleaning
    d[ycol] = _to_num(d[ycol]).fillna(0).astype(int)
    d[wcol] = _to_num(d[wcol]).fillna(1.0)

    # admission-only defaults
    cont_mean_sd = [c for c in ["age", "i10_ndx"] if c in d.columns]
    cont_med_iqr = []  # none for admission-only
    if include_course_vars:
        cont_med_iqr = [c for c in ["los", "i10_npr", "totchg"] if c in d.columns]

    cat_vars = [c for c in [
        "female","pay1","zipinc_qrtl","pl_nchs","aweekend","elective","hcup_ed",
        "sepsis","aki","stroke","chf","ckd","uti","pneumonia","delirium","copd"
    ] if c in d.columns]

    # numeric conversions
    for c in cont_mean_sd + cont_med_iqr:
        d[c] = _to_num(d[c])

    # binary cleanup if present
    for c in ["female","aweekend","elective","sepsis","aki","stroke","chf","ckd","uti","pneumonia","delirium","copd"]:
        if c in d.columns:
            # keep 0/1 if possible
            s = _to_num(d[c])
            if s.dropna().isin([0,1]).all():
                d[c] = s.fillna(0).astype(int)

    d0 = d[d[ycol]==0].copy()
    d1 = d[d[ycol]==1].copy()

    W_all = d[wcol].sum()
    W_0   = d0[wcol].sum()
    W_1   = d1[wcol].sum()

    rows = []
    rows.append({"Variable":"Weighted N (sum of discwt)", "Overall":f"{W_all:,.0f}", "Survived":f"{W_0:,.0f}", "Died":f"{W_1:,.0f}", "p-value":""})
    rows.append({"Variable":"Unweighted N", "Overall":f"{len(d):,}", "Survived":f"{len(d0):,}", "Died":f"{len(d1):,}", "p-value":""})

    # continuous mean(SD)
    for v in cont_mean_sd:
        da = d[[v,wcol]].dropna(); d0a = d0[[v,wcol]].dropna(); d1a = d1[[v,wcol]].dropna()
        m_all, sd_all = weighted_mean(da[v], da[wcol]), weighted_sd(da[v], da[wcol])
        m_0, sd_0     = weighted_mean(d0a[v], d0a[wcol]), weighted_sd(d0a[v], d0a[wcol])
        m_1, sd_1     = weighted_mean(d1a[v], d1a[wcol]), weighted_sd(d1a[v], d1a[wcol])
        p = p_unweighted_t(d, v)
        rows.append({"Variable":v, "Overall":fmt_mean_sd(m_all, sd_all), "Survived":fmt_mean_sd(m_0, sd_0), "Died":fmt_mean_sd(m_1, sd_1),
                     "p-value":("" if not np.isfinite(p) else ("<0.001" if p<0.001 else f"{p:.3f}"))})

    # continuous median(IQR) (course-only)
    for v in cont_med_iqr:
        da = d[[v,wcol]].dropna(); d0a = d0[[v,wcol]].dropna(); d1a = d1[[v,wcol]].dropna()
        q_all = weighted_quantile(da[v], [0.25,0.50,0.75], da[wcol])
        q_0   = weighted_quantile(d0a[v],[0.25,0.50,0.75], d0a[wcol])
        q_1   = weighted_quantile(d1a[v],[0.25,0.50,0.75], d1a[wcol])
        rows.append({"Variable":v, "Overall":fmt_med_iqr(q_all[1], q_all[0], q_all[2]),
                     "Survived":fmt_med_iqr(q_0[1], q_0[0], q_0[2]),
                     "Died":fmt_med_iqr(q_1[1], q_1[0], q_1[2]),
                     "p-value":""})

    # categorical (weighted n, %)
    for v in cat_vars:
        rows.append({"Variable":v, "Overall":"", "Survived":"", "Died":"", "p-value":""})
        wc_all, tot_all = weighted_cat_counts(d, v)
        wc_0, tot_0     = weighted_cat_counts(d0, v)
        wc_1, tot_1     = weighted_cat_counts(d1, v)

        levels = sorted(set(wc_all.index.tolist()) | set(wc_0.index.tolist()) | set(wc_1.index.tolist()))
        p = p_unweighted_chi2(d, v)
        p_fmt = "" if not np.isfinite(p) else ("<0.001" if p<0.001 else f"{p:.3f}")

        for lv in levels:
            n_all = wc_all.get(lv, 0.0); pct_all = (n_all/tot_all*100) if tot_all>0 else np.nan
            n_0   = wc_0.get(lv, 0.0);   pct_0   = (n_0/tot_0*100) if tot_0>0 else np.nan
            n_1   = wc_1.get(lv, 0.0);   pct_1   = (n_1/tot_1*100) if tot_1>0 else np.nan
            rows.append({
                "Variable": f"  {label_level(v, lv)}",
                "Overall": fmt_n_pct(n_all, pct_all) if np.isfinite(pct_all) else "",
                "Survived": fmt_n_pct(n_0, pct_0) if np.isfinite(pct_0) else "",
                "Died": fmt_n_pct(n_1, pct_1) if np.isfinite(pct_1) else "",
                "p-value": p_fmt
            })

    out = pd.DataFrame(rows)
    out["Variable"] = out["Variable"].replace(rename_map)
    return out

# =========================
# GENERATE TABLE 1 + SUPP TABLE
# =========================
table1  = build_table1(df, include_course_vars=False)  # admission-only (recommended as Table 1)
tableS1 = build_table1(df, include_course_vars=True)   # includes LOS/procedures/charges (supplement)

display(table1)
print("\n--- Supplementary (includes LOS/procedures/charges) ---\n")
display(tableS1)

table1.to_csv("Table1_admission_only_weighted.csv", index=False)
tableS1.to_csv("TableS1_full_course_weighted.csv", index=False)
print("\nSaved: Table1_admission_only_weighted.csv")
print("Saved: TableS1_full_course_weighted.csv")


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Inputs you already have
# df = your full NRD dataframe (already loaded)
# dx_cols = list of diagnosis columns, e.g., i10_dx1...i10_dx30
# -----------------------------

# Example auto-detect dx columns if needed:
dx_cols = [c for c in df.columns if c.lower().startswith("i10_dx")]
dx_cols = sorted(dx_cols)

# Ensure LOS numeric
df["los"] = pd.to_numeric(df["los"], errors="coerce")

# Build LOS groups
short = df[df["los"] <= 5].copy()
long  = df[df["los"] >= 25].copy()

def top_icd_families(sub_df, dx_cols, top_n=15):
    """
    Counts % of admissions having each ICD-10 3-char prefix family
    (e.g., F03, E78, I10). Counts each admission once per family.
    """
    N = len(sub_df)

    # Long format: one dx code per row, keep admission index
    long_dx = (
        sub_df[dx_cols]
        .astype("string")
        .stack()
        .reset_index(level=1, drop=True)     # keep original admission index
        .rename("icd")
        .str.upper()
    )

    # Drop missing/blank
    long_dx = long_dx.dropna()
    long_dx = long_dx[long_dx.str.len() >= 3]

    # ICD family prefix (first 3 chars)
    fam = long_dx.str.slice(0, 3)

    # Count admissions with each family (unique per admission)
    fam_per_adm = pd.DataFrame({"fam": fam}).reset_index().drop_duplicates()
    counts = fam_per_adm["fam"].value_counts()

    out = pd.DataFrame({
        "ICD10_family": counts.index,
        "n_admissions": counts.values,
    })
    out["percent"] = 100 * out["n_admissions"] / N
    out = out.head(top_n)

    return N, out

N_short, tab_short = top_icd_families(short, dx_cols, top_n=20)
N_long, tab_long   = top_icd_families(long,  dx_cols, top_n=20)

print("Short LOS (≤5 days) N =", N_short)
display(tab_short)

print("Prolonged LOS (≥25 days) N =", N_long)
display(tab_long)
